# End-to-end workflow: hourly crypto algo-trading
Every step calls a function from `src/*.py`. Set `RUN_HEAVY=True` to re-run download, training and the simulation grid (about 15 min); otherwise the notebook reuses the files on disk.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src import download, transform, eda, train, simulate, predict, evaluate
RUN_HEAVY = False

## 1. Download (incremental)
Only missing months/days are fetched from data.binance.vision.

In [2]:
download.run(symbols=None if RUN_HEAVY else ['BTCUSDT'], start='2026-10')

BTCUSDT    +0 klines  +0 funding
fear_greed rows: 3210


## 2. Transform: one table, features, targets, splits

In [3]:
df = transform.run()
fs = transform.get_feature_sets(df)
{k: len(v) for k, v in fs.items()}

dataset: (1305006, 76)  symbols=50
split
train    719139
test     369744
valid    212595
purge      2328
live       1200
numeric=32  dummies=21


{'TO_PREDICT': 5, 'NUMERIC': 32, 'DUMMIES': 21, 'FEATURES': 53}

## 3. EDA

In [4]:
eda.run()

ret_fwd_24h describe:
 count    719139.0000
mean          0.0012
std           0.0518
min          -0.4981
1%           -0.1285
5%           -0.0747
50%          -0.0004
95%           0.0826
99%           0.1527
max           1.0299

P(up)=0.493   P(>1%)=0.386

Spearman correlation with targets (train):
                      ret_fwd_24h    y_up
fg_chg_7d                -0.0511 -0.0436
fear_greed               -0.0424 -0.0393
btc_ret_24h              -0.0401 -0.0440
ret_168h                 -0.0293 -0.0391
close_sma_24             -0.0289 -0.0349
ret_72h                  -0.0287 -0.0374
ret_12h                  -0.0284 -0.0335
macd_hist                -0.0261 -0.0237
ret_24h                  -0.0252 -0.0324
close_sma_168            -0.0233 -0.0350
rel_ret_24h              -0.0232 -0.0163
bb_pos                   -0.0219 -0.0274
btc_ret_1h               -0.0198 -0.0181
dist_high_168h           -0.0196 -0.0256
ret_4h                   -0.0187 -0.0198
rank_ret_24h             -0.0159 -0.01

## 4. Modeling (LR, DecisionTree, RandomForest, XGBoost + tuning)

In [5]:
if RUN_HEAVY:
    train.run(n_iter=5)
pd.read_csv(ROOT/'data/processed/model_summary.csv')[['model','valid_auc','test_auc']]

,model,valid_auc,test_auc
0,lr,0.482877,0.514513
1,dt,0.510674,0.508993
2,rf,0.519594,0.512909
3,xgb,0.520054,0.523135


## 5. Trading simulation vs benchmarks

In [6]:
if RUN_HEAVY:
    simulate.run()
    simulate.run_daily()
import json
json.loads((ROOT/'reports/best_strategy.json').read_text())

{'strategy': {'total_return': -0.40917757092385565,
  'cagr': -0.4503370457423369,
  'sharpe': -1.232007374118458,
  'max_drawdown': -0.4733666809464879,
  'calmar': -0.9513492686935555,
  'trades': 346,
  'win_rate': 0.4479768786127168,
  'avg_trade': -0.004233602950481475},
 'benchmarks': {'BTC buy&hold': {'total_return': -0.26035845279693604,
   'cagr': -0.2903429865837097,
   'sharpe': -0.5939589292590645,
   'max_drawdown': -0.40310269594192505,
   'calmar': -0.7202705144882202},
  'Equal-weight basket': {'total_return': -0.3548622727394104,
   'cagr': -0.39251846075057983,
   'sharpe': -0.5356596235551098,
   'max_drawdown': -0.5144863128662109,
   'calmar': -0.7629327774047852}},
 'config': {'model': 'pred_dt',
  'n_slots': 3,
  'q': 0.95,
  'long_short': True,
  'sl': None,
  'prob_sizing': False},
 'thr_hi': 0.879,
 'thr_lo': 0.12857142857142856}

## 6. Error analysis by symbol / month / hour / regime

In [7]:
_ = evaluate.run('pred_xgb')


== pred_xgb by symbol
              rows    auc  hit_top10   base   edge  avg_ret_top10
symbol                                                          
TRXUSDT    7704.0  0.503      0.494  0.530 -0.036          0.001
BNBUSDT    7704.0  0.498      0.503  0.508 -0.005         -0.002
RUNEUSDT   7704.0  0.502      0.477  0.479 -0.002         -0.001
DOGEUSDT   7704.0  0.523      0.458  0.456  0.002         -0.000
KAVAUSDT   7680.0  0.478      0.510  0.502  0.008         -0.004
SOLUSDT    7704.0  0.519      0.495  0.487  0.008         -0.000
AAVEUSDT   7704.0  0.540      0.490  0.478  0.012          0.000
DOTUSDT    7704.0  0.499      0.446  0.433  0.013         -0.003
EGLDUSDT   7704.0  0.492      0.462  0.445  0.017          0.001
ALGOUSDT   7704.0  0.494      0.472  0.449  0.023          0.004
NEARUSDT   7704.0  0.519      0.508  0.473  0.035          0.004
SHIBUSDT   7704.0  0.519      0.473  0.437  0.037          0.001
ETCUSDT    7704.0  0.516      0.470  0.433  0.037         -0.002
B

d:\Git Repo\MLZoomcamp\stock-market-analytics\src\evaluate.py:62: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print("\nWorst high-confidence calls:\n", worst.round(3).to_string(index=False))


## 7. Today's target book (daily market-neutral strategy)

In [8]:
book = predict.predict_live()
print(predict.format_message(book))
book

Daily market-neutral book as of 2026-11-18 23:00 UTC
Positions: 10 long / 10 short (equal weights, 50% capital per side)
Trades to make:
  CLOSE BNBUSDT    -     @ 582.71
  CLOSE LINKUSDT   -     @ 9.083
  CLOSE CHZUSDT    -     @ 0.00973
  OPEN  ETCUSDT    long  @ 5.93
  OPEN  BCHUSDT    long  @ 208.7
  OPEN  SNXUSDT    short @ 0.1566
Unchanged: KAVAUSDT, SOLUSDT, XRPUSDT, DOGEUSDT, SUIUSDT, UNIUSDT, ETHUSDT, BTCUSDT, ATOMUSDT, IMXUSDT, EGLDUSDT, MANAUSDT, THETAUSDT, ENJUSDT, INJUSDT, ICPUSDT, TRXUSDT


,symbol,as_of,signal,weight,prev_weight,price,side,action
2,BNBUSDT,2026-11-18 23:00:00+00:00,0.002210,0.00,0.05,582.710022,-,CLOSE
14,LINKUSDT,2026-11-18 23:00:00+00:00,0.001851,0.00,0.05,9.083000,-,CLOSE
4,CHZUSDT,2026-11-18 23:00:00+00:00,-0.001049,0.00,-0.05,0.009730,-,CLOSE
13,KAVAUSDT,2026-11-18 23:00:00+00:00,0.005292,0.05,0.05,0.028470,long,hold
17,SOLUSDT,2026-11-18 23:00:00+00:00,0.004879,0.05,0.05,73.639999,long,hold
22,XRPUSDT,2026-11-18 23:00:00+00:00,0.003006,0.05,0.05,0.967700,long,hold
5,DOGEUSDT,2026-11-18 23:00:00+00:00,0.002814,0.05,0.05,0.058780,long,hold
18,SUIUSDT,2026-11-18 23:00:00+00:00,0.002746,0.05,0.05,0.720200,long,hold
21,UNIUSDT,2026-11-18 23:00:00+00:00,0.002740,0.05,0.05,5.187000,long,hold
8,ETCUSDT,2026-11-18 23:00:00+00:00,0.002681,0.05,0.00,5.930000,long,OPEN
